# 03_sarima_store_level

Теперь переходим к классическому статистическому подходу для временных рядов.
SARIMA здесь выступает не как универсальное решение для всей задачи,
а как содержательный reference: модель, которая умеет работать с трендом и сезонностью,
но при этом требует отдельной подгонки для каждой серии.

Из-за вычислительной тяжести мы ограничиваемся наиболее значимыми рядами.
Такой режим позволяет сохранить методологический смысл сравнения,
не превращая расчет в слишком дорогой полный перебор по всей панели.


In [ ]:
import os
import time
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import pmdarima as pm

from concurrent.futures import ProcessPoolExecutor, as_completed
import warnings

warnings.filterwarnings("ignore")

# Настройка отображения pandas
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Настройка логгера
logger = logging.getLogger("sarima_store_level")
logger.setLevel(logging.INFO)

# если в ноутбуке логгер уже был настроен, не дублируем хендлеры
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger sarima_store_level initialized")


In [ ]:
from pathlib import Path
import json
import pickle

PROJECT_ROOT = Path('..')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DATASETS_DIR = ARTIFACTS_DIR / 'datasets'
METADATA_DIR = ARTIFACTS_DIR / 'metadata'
METRICS_DIR = ARTIFACTS_DIR / 'metrics'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'

for path in [METRICS_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

train_processed = pd.read_csv(DATASETS_DIR / 'train_processed.csv', parse_dates=['date'])
with open(METADATA_DIR / 'splits.json', 'r', encoding='utf-8') as f:
    splits_json = json.load(f)
top_pairs_df = pd.read_csv(DATASETS_DIR / 'top_pairs.csv')

top_pairs = set(top_pairs_df[['store_nbr', 'family']].itertuples(index=False, name=None))
splits = []
for s in splits_json:
    splits.append({
        'name': s['name'],
        'train_idx': pd.Index(s['train_idx']),
        'val_idx': pd.Index(s['val_idx']),
        'train_end': pd.to_datetime(s['train_end']),
        'val_start': pd.to_datetime(s['val_start']),
        'val_end': pd.to_datetime(s['val_end']),
    })

logger.info(f'Loaded train_processed: {train_processed.shape}')
logger.info(f'Loaded splits: {len(splits)}')
logger.info(f'Loaded top_pairs: {len(top_pairs)}')


## Протокол эксперимента

Здесь особенно важно не смешивать разные варианты постановки задачи.
Мы фиксируем один набор временных сплитов и одно подмножество серий,
после чего оцениваем SARIMA последовательно и в одинаковых условиях для всех окон валидации.


In [ ]:
def filter_to_top_pairs(df):
    mask = df[['store_nbr', 'family']].apply(tuple, axis=1).isin(top_pairs)
    return df[mask].copy()

splits_overview = []
for s in splits:
    tr = train_processed.loc[s['train_idx']]
    va = train_processed.loc[s['val_idx']]
    tr_top = filter_to_top_pairs(tr)
    va_top = filter_to_top_pairs(va)
    splits_overview.append({
        'split': s['name'],
        'train_start': tr['date'].min(),
        'train_end': tr['date'].max(),
        'val_start': va['date'].min(),
        'val_end': va['date'].max(),
        'train_rows_top_pairs': len(tr_top),
        'val_rows_top_pairs': len(va_top),
        'n_pairs_top_pairs': tr_top[['store_nbr', 'family']].drop_duplicates().shape[0],
    })

pd.DataFrame(splits_overview)


## Ядро локальной SARIMA-модели

Ниже собраны функции для работы с одной серией:
извлечение ряда, fallback-логика и обучение самой SARIMA.
Идея в том, чтобы сначала аккуратно определить поведение модели на локальном уровне,
а уже потом масштабировать этот подход на набор серий.


In [ ]:
def get_series_for_pair_store(train_df, store_nbr, family):
    """
    Возвращает временной ряд продаж для пары (store_nbr, family).

    Формат:
    - индекс: date (отсортирован)
    - значения: daily sales (float32)
    """
    sub = train_df[
        (train_df["store_nbr"] == store_nbr) &
        (train_df["family"] == family)
    ]

    if sub.empty:
        return pd.Series(dtype="float32")

    ts = (
        sub.groupby("date")["sales"]
        .sum()   # или .mean(), но для одного магазина/семьи sum == исходным продажам
        .sort_index()
        .astype("float32")
    )
    return ts


In [ ]:
def fallback_forecast_recursive(ts: pd.Series, horizon: int = 16) -> np.ndarray:
    """
    Простой рекурсивный эвристический прогноз на horizon дней вперёд.

    Идея:
    - медиана последних 30 значений (история + уже спрогнозированные);
    - уровень недели назад или последнее значение;
    - прогноз = среднее этих двух оценок, обрезанное снизу нулём.
    """
    ts = pd.Series(ts).sort_index().astype("float64")

    if ts.empty:
        return np.zeros(horizon, dtype="float32")

    values = ts.to_numpy().tolist()
    y_hat = np.empty(horizon, dtype="float32")

    for h in range(horizon):
        window = values[-30:]
        med_30 = float(np.nanmedian(window))

        if len(values) >= 7:
            seasonal_level = values[-7]
        else:
            seasonal_level = values[-1]

        base = 0.5 * med_30 + 0.5 * seasonal_level
        base = 0.0 if not np.isfinite(base) else max(base, 0.0)

        y_hat[h] = base
        values.append(base)

    return y_hat


In [ ]:
def fit_sarima_one_series(y_train: pd.Series):
    """
    Обучение SARIMA (auto_arima, pmdarima) для одного ряда продаж.

    Параметры подобраны консервативно, чтобы не взорвать время.
    """
    logger = logging.getLogger("sarima_store_level")

    y = pd.Series(y_train).astype("float64").fillna(0.0).clip(lower=0.0)

    # минимальные требования к длине ряда
    if len(y) < 30 or y.sum() == 0:
        logger.info("Series too short or zero-sum, skipping SARIMA")
        return None

    if y.isna().any():
        logger.warning(
            f"Series has NaN after preprocessing: {y.isna().sum()} values"
        )
        return None

    try:
        model = pm.auto_arima(
            y,
            seasonal=True,
            m=7,              # недельная сезонность
            start_p=0, max_p=2,
            start_q=0, max_q=2,
            d=None,           # авто-выбор d
            start_P=0, max_P=1,
            start_Q=0, max_Q=1,
            D=None,           # авто-выбор D
            error_action="ignore",
            suppress_warnings=True,
            stepwise=True,
            max_order=5,
            trace=False,
        )
        logger.info(
            f"auto_arima fitted: order={model.order}, "
            f"seasonal_order={model.seasonal_order}"
        )
        return model
    except Exception as e:
        logger.warning(f"auto_arima failed: {e}")
        return None


In [ ]:
def forecast_sarima(model, horizon: int = 16) -> np.ndarray:
    """
    Возвращает прогноз на horizon дней вперёд для одной серии.

    Если model is None или predict падает, возвращает нули.
    """
    if model is None:
        return np.zeros(horizon, dtype="float32")

    try:
        y_forecast = model.predict(n_periods=horizon)
        y_forecast = np.array(y_forecast, dtype="float32")
        y_forecast = np.clip(y_forecast, 0.0, None)
        return y_forecast
    except Exception as e:
        logger = logging.getLogger("sarima_store_level")
        logger.warning(f"SARIMA predict failed: {e}")
        return None


## Параллельный прогон по `(store_nbr, family)`

Основная вычислительная сложность SARIMA проявляется именно здесь:
модель приходится подбирать отдельно для каждой серии.
Поэтому дальнейший расчет имеет смысл организовать параллельно,
иначе даже ограничение на значимые ряды окажется слишком медленным.


In [ ]:
GLOBAL_TRAIN_DF = None

def init_worker(train_df):
    """
    Инициализатор воркера: сохраняем train_df в глобальной переменной,
    чтобы не передавать тяжёлый DataFrame в каждый submit.
    """
    global GLOBAL_TRAIN_DF
    GLOBAL_TRAIN_DF = train_df
    logger = logging.getLogger("sarima_store_level")
    logger.info("Worker initialized with GLOBAL_TRAIN_DF")


In [ ]:
def run_sarima_for_pair_store(store_nbr, family, val_start, val_end):
    """
    Строит SARIMA-прогноз для одной пары (store_nbr, family) на валидационном окне.

    Возвращает DataFrame:
    - date
    - store_nbr
    - family
    - y_pred
    длиной = числу дней в интервале [val_start, val_end].
    """
    logger = logging.getLogger("sarima_store_level")

    global GLOBAL_TRAIN_DF
    if GLOBAL_TRAIN_DF is None:
        raise RuntimeError("GLOBAL_TRAIN_DF is not initialized in worker")

    pid = os.getpid()

    ts_train = get_series_for_pair_store(GLOBAL_TRAIN_DF, store_nbr, family)

    horizon = (val_end - val_start).days + 1
    dates = pd.date_range(val_start, val_end, freq="D")

    # если ряд пустой/слишком короткий/нулевой — сразу фолбэк
    if ts_train.empty or len(ts_train) < 10 or ts_train.sum() == 0:
        logger.info(
            f"[PID={pid}] fallback for (store={store_nbr}, family={family}): "
            f"empty/short/zero series"
        )
        y_hat = fallback_forecast_recursive(ts_train, horizon=horizon)
    else:
        logger.info(
            f"[PID={pid}] start (store={store_nbr}, family={family})"
        )
        try:
            model = fit_sarima_one_series(ts_train)
            if model is None:
                y_hat = fallback_forecast_recursive(ts_train, horizon=horizon)
                logger.info(
                    f"[PID={pid}] SARIMA skipped, using fallback for "
                    f"(store={store_nbr}, family={family})"
                )
            else:
                y_hat = forecast_sarima(model, horizon=horizon)
                
                if y_hat is None:
                    y_hat = fallback_forecast_recursive(ts_train, horizon=horizon)
                    logger.info(
                        f"[PID={pid}] SARIMA skipped, using fallback for "
                        f"(store={store_nbr}, family={family})"
                    )
                else:
                    logger.info(
                        f"[PID={pid}] SARIMA OK for (store={store_nbr}, family={family}), "
                        f"horizon={horizon}"
                    )
        except Exception as e:
            logger.warning(
                f"[PID={pid}] run_sarima_for_pair_store failed for "
                f"(store={store_nbr}, family={family}): {e}. Using fallback."
            )
            y_hat = fallback_forecast_recursive(ts_train, horizon=horizon)

    df_pred = pd.DataFrame({
        "date": dates,
        "store_nbr": store_nbr,
        "family": family,
        "y_pred": y_hat.astype("float32"),
    })

    return df_pred


In [ ]:
def predict_sarima_on_split(
    train_df,
    val_df,
    split_name,
    max_workers=4,
    pairs_subset=None,
):
    """
    Строит SARIMA-прогнозы для всех (или выбранных) пар (store_nbr, family)
    на заданном сплите.

    Параметры:
    - train_df: train-часть сплита
    - val_df:   val-часть сплита
    - split_name: имя сплита для логов
    - max_workers: число воркеров в ProcessPoolExecutor
    - pairs_subset: опциональный список/сет пар (store_nbr, family),
      если хотим ограничиться подмножеством.

    Возвращает:
    - preds_df: DataFrame с колонками [date, store_nbr, family, y_pred]
      на всём валидационном окне.
    """
    logger = logging.getLogger("sarima_store_level")

    split_start_time = time.time()
    split_start_dt = datetime.now()
    logger.info(
        f"[{split_name}] Start SARIMA prediction on split "
        f"at {split_start_dt.strftime('%Y-%m-%d %H:%M:%S')}"
    )

    val_start = val_df["date"].min()
    val_end = val_df["date"].max()
    logger.info(
        f"[{split_name}] Validation window: {val_start.date()} → {val_end.date()}"
    )

    # полный список пар в train
    pairs = (
        train_df[["store_nbr", "family"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    pairs = list(pairs)

    # если передан pairs_subset, фильтруем список пар
    if pairs_subset is not None:
        pairs_set = set(pairs_subset)
        pairs = [p for p in pairs if p in pairs_set]

    n_pairs = len(pairs)
    logger.info(f"[{split_name}] Number of (store, family) pairs: {n_pairs}")

    results = []
    n_ok = 0
    n_fail = 0
    total_pair_time = 0.0

    logger.info(
        f"[{split_name}] Launching ProcessPoolExecutor with max_workers={max_workers}"
    )

    with ProcessPoolExecutor(
        max_workers=max_workers,
        initializer=init_worker,
        initargs=(train_df,),
    ) as executor:
        future_to_meta = {}
        for (store_nbr, family) in pairs:
            submit_time = time.time()
            fut = executor.submit(
                run_sarima_for_pair_store,
                store_nbr,
                family,
                val_start,
                val_end,
            )
            future_to_meta[fut] = (store_nbr, family, submit_time)

        for i, future in enumerate(as_completed(future_to_meta), start=1):
            store_nbr, family, submit_time = future_to_meta[future]
            pair_start_time = submit_time

            try:
                df_pred = future.result()
                n_ok += 1
                pair_elapsed = time.time() - pair_start_time
                total_pair_time += pair_elapsed
                logger.info(
                    f"[{split_name}] Pair (store={store_nbr}, family={family}) "
                    f"finished in {pair_elapsed:.2f} seconds "
                    f"(queue + fit+forecast)"
                )
            except Exception as exc:
                n_fail += 1
                logger.warning(
                    f"[{split_name}] Failed pair (store={store_nbr}, family={family}): {exc}"
                )
                pair_elapsed = time.time() - pair_start_time
                total_pair_time += pair_elapsed
                logger.info(
                    f"[{split_name}] Failed pair (store={store_nbr}, family={family}) "
                    f"took {pair_elapsed:.2f} seconds before failure"
                )

                dates = pd.date_range(val_start, val_end, freq="D")
                df_pred = pd.DataFrame({
                    "date": dates,
                    "store_nbr": store_nbr,
                    "family": family,
                    "y_pred": np.zeros(len(dates), dtype="float32"),
                })

            results.append(df_pred)

            if i % 20 == 0 or i == n_pairs:
                avg_pair_time = total_pair_time / i
                logger.info(
                    f"[{split_name}] Progress: {i}/{n_pairs} pairs processed "
                    f"(ok={n_ok}, fail={n_fail}), "
                    f"avg pair time so far = {avg_pair_time:.2f} seconds "
                    f"(queue + fit+forecast)"
                )

    split_elapsed = time.time() - split_start_time
    avg_pair_time_overall = total_pair_time / max(n_pairs, 1)
    logger.info(
        f"[{split_name}] All pairs processed in {split_elapsed:.2f} seconds "
        f"(avg per pair = {avg_pair_time_overall:.2f} seconds, "
        f"queue + fit+forecast)"
    )

    preds_df = pd.concat(results, ignore_index=True)
    logger.info(f"[{split_name}] preds_df shape: {preds_df.shape}")

    return preds_df


## Метрики и сохранение результатов

В конце нам нужно получить не просто одно число качества,
а аккуратную сводку по всем сплитам и набор предсказаний для последующего сравнения.
Это важно, потому что дальше SARIMA должна сравниваться с другими подходами уже не на уровне впечатлений,
а на уровне одинаково посчитанных метрик.


In [ ]:
def calculate_metrics(y_true, y_pred):
    """
    Расчёт базовых метрик:
    - RMSLE
    - RMSE
    - MAE
    - WAPE (weighted absolute percentage error)

    Прогноз обрезается снизу нулём, если пришёл отрицательным.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_pred = np.clip(y_pred, 0, None)

    # RMSLE
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

    # RMSE
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # MAE
    mae = np.mean(np.abs(y_true - y_pred))

    # WAPE
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        wape = np.nan
    else:
        wape = np.sum(np.abs(y_true - y_pred)) / denom

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "WAPE": wape,
    }


In [ ]:
def evaluate_on_split_store_level(preds_df, val_df, split_name):
    """
    Считает метрики на уровне (store_nbr, family, date) для заданного сплита.

    Параметры:
    - preds_df: DataFrame с колонками [date, store_nbr, family, y_pred],
      результат работы predict_sarima_on_split.
    - val_df: DataFrame валидационной части (исходный train_processed с колонкой sales).
    - split_name: имя сплита (для логов и результата).

    Возвращает:
    - metrics: dict с ключами RMSLE, RMSE, MAE, WAPE, split, model.
    - val_pred_df: DataFrame со слитыми y_true и y_pred для дальнейшего анализа.
    """
    logger = logging.getLogger("sarima_store_level")

    # Берём из val_df фактические продажи на уровне (date, store_nbr, family)
    # У нас в val_df уже есть колонка sales, просто переименуем её
    val_true = val_df[["date", "store_nbr", "family", "sales"]].copy()
    val_true = val_true.rename(columns={"sales": "y_true"})

    # Слияние
    val_pred_df = preds_df.merge(
        val_true,
        on=["date", "store_nbr", "family"],
        how="left",
    )

    before_filter = len(val_pred_df)
    val_pred_df = val_pred_df[~val_pred_df["y_true"].isna()].copy()
    after_filter = len(val_pred_df)

    logger.info(
        f"[{split_name}] val_pred_df filtered: {before_filter} → {after_filter} "
        f"rows with y_true"
    )

    y_true = val_pred_df["y_true"].values
    y_pred = val_pred_df["y_pred"].values

    metrics = calculate_metrics(y_true, y_pred)
    metrics["split"] = split_name
    metrics["model"] = "sarima_store_family"

    logger.info(
        f"[{split_name}] Metrics: "
        f"RMSLE={metrics['RMSLE']:.4f}, "
        f"RMSE={metrics['RMSE']:.2f}, "
        f"MAE={metrics['MAE']:.2f}, "
        f"WAPE={metrics['WAPE']:.4f}"
    )

    return metrics, val_pred_df


In [ ]:
sarima_metrics_by_split = {}
sarima_preds_by_split = {}

MAX_WORKERS = 3  # при необходимости скорректировать

for s in splits:
    split_name = s["name"]
    logger.info(f"=== {split_name} ===")

    # Train/val для сплита
    train_df_split = train_processed.loc[s["train_idx"]]
    val_df_split   = train_processed.loc[s["val_idx"]]

    # Фильтрация по top_pairs
    train_df_top = filter_to_top_pairs(train_df_split)
    val_df_top   = filter_to_top_pairs(val_df_split)

    logger.info(
        f"{split_name} (top_pairs) | "
        f"train: {len(train_df_top)} rows, "
        f"val: {len(val_df_top)} rows"
    )

    # Список пар для этого сплита (все top_pairs, которые реально есть в train)
    pairs_all = (
        train_df_top[["store_nbr", "family"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    pairs_all = list(pairs_all)
    logger.info(f"{split_name} (top_pairs) | total pairs: {len(pairs_all)}")

    # Прогон SARIMA по всем этим парам
    preds_df = predict_sarima_on_split(
        train_df=train_df_top,
        val_df=val_df_top,
        split_name=split_name,
        max_workers=MAX_WORKERS,
        pairs_subset=pairs_all,  # можно и не передавать, но так явнее
    )

    # Оценка качества
    metrics, val_pred_df = evaluate_on_split_store_level(
        preds_df,
        val_df_top,
        split_name=split_name,
    )

    # Сохраняем
    sarima_metrics_by_split[split_name] = metrics
    sarima_preds_by_split[split_name] = val_pred_df


In [ ]:
# Преобразуем словарь метрик в DataFrame
metrics_rows = []
for split_name, m in sarima_metrics_by_split.items():
    row = {"split": split_name}
    row.update({
        "RMSLE": m["RMSLE"],
        "RMSE": m["RMSE"],
        "MAE": m["MAE"],
        "WAPE": m["WAPE"],
    })
    metrics_rows.append(row)

sarima_metrics_df = pd.DataFrame(metrics_rows).set_index("split")

# Добавим строку со средними по всем сплитам
mean_row = sarima_metrics_df.mean(axis=0).to_dict()
mean_row = pd.DataFrame(mean_row, index=["mean"])
sarima_metrics_df = pd.concat([sarima_metrics_df, mean_row])

sarima_metrics_df


In [ ]:
metrics_path = METRICS_DIR / 'sarima_store_metrics_by_split.csv'
sarima_metrics_df.to_csv(metrics_path, index=True)
logger.info(f'SARIMA metrics saved to {metrics_path}')

preds_path = PREDICTIONS_DIR / 'sarima_store_preds_by_split.pkl'
with open(preds_path, 'wb') as f:
    pickle.dump(sarima_preds_by_split, f)
logger.info(f'SARIMA predictions by split saved to {preds_path}')


In [ ]:
sarima_metrics_df
